# HEALPix Pixelization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CMBSciPol/jax-healpy/blob/main/docs/user-guide/pixelization.ipynb)

This page details the parts of the HEALPix scheme that [Getting started](getting-started.md)
goes over quickly: the two orders of the pixels, the 12 base faces, the astronomical
coordinates, the interpolation between the pixels and the choice of the resolution. The
[HEALPix primer](https://healpix.sourceforge.io/pdf/intro.pdf) and
[Górski et al. (2005)](https://doi.org/10.1086/427976) describe the scheme.

In [ ]:
# Install jax-healpy and its dependencies on Google Colab. A CPU runtime, the default one, is expected:
# the `cpu` extra installs ducc0, which computes the spherical harmonic transforms on CPU only.
import sys

if 'google.colab' in sys.modules:
    %pip install --quiet "jax-healpy[cpu] @ git+https://github.com/CMBSciPol/jax-healpy" healpy

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from healpy import graticule, mollview, projscatter, projtext

import jax_healpy as hp

jax.config.update('jax_enable_x64', True)

## RING and NESTED orders

Each pixel of an `nside = 2` map (48 pixels) shows its index in both orders. In the RING
order, the indices increase along the rings of constant latitude, from the North pole to the
South pole. In the NESTED order, the 4 pixels of each of the 12 base pixels have consecutive
indices.

In [ ]:
nside_demo = 2
npix_demo = hp.nside2npix(nside_demo)
ring_indices = jnp.arange(npix_demo)
theta_demo, phi_demo = hp.pix2ang(nside_demo, ring_indices)

fig = plt.figure(figsize=(12, 3.5))
for sub, (name, nest) in enumerate([('RING', False), ('NESTED', True)], start=1):
    # Value of each pixel (in RING order) = its index in the order
    indices = ring_indices if not nest else hp.ring2nest(nside_demo, ring_indices)
    mollview(
        indices,
        fig=fig.number,
        sub=(1, 2, sub),
        title=f'{name} order',
        cmap='Blues',
        min=-npix_demo / 2,
        max=npix_demo - 1,
        cbar=False,
    )
    for t, p, index in zip(theta_demo, phi_demo, indices):
        projtext(t, p, str(index), ha='center', va='center', fontsize=8)
plt.show()

The NESTED order makes the hierarchy explicit: at `2 * nside`, the pixel of index `p` is
divided into the 4 pixels of indices `4 * p` to `4 * p + 3`. Changing the resolution is then an
integer division of the indices:

In [ ]:
parent = 5
children = 4 * parent + jnp.arange(4)
theta_children, phi_children = hp.pix2ang(2 * nside_demo, children, nest=True)
print(
    f'Pixels at nside = {nside_demo} that contain the children: {hp.ang2pix(nside_demo, theta_children, phi_children, nest=True)}'
)

`ring2nest` and `nest2ring` convert the indices of pixels, and `reorder` converts whole maps:

In [ ]:
nside = 32
ring_pixels = jnp.arange(100)
nest_pixels = hp.ring2nest(nside, ring_pixels)
print(f'Round trip of the indices: {jnp.array_equal(ring_pixels, hp.nest2ring(nside, nest_pixels))}')

ring_map = jax.random.normal(jax.random.key(0), (hp.nside2npix(nside),))
nest_map = hp.reorder(ring_map, r2n=True)
print(f'Round trip of the map: {jnp.array_equal(ring_map, hp.reorder(nest_map, n2r=True))}')

## Base faces

The 12 base pixels, or faces, tile the sphere: 4 around the North pole, 4 along the equator and
4 around the South pole. Every pixel, at any resolution, belongs to one of them. `pix2xyf` gives
the face of a pixel and its position in the face, and `xyf2pix` converts back:

In [ ]:
pixels = jnp.arange(8)
x, y, face = hp.pix2xyf(nside, pixels)
for pixel, xi, yi, fi in zip(pixels, x, y, face):
    print(f'Pixel {pixel}: x = {xi:2d}, y = {yi:2d}, face {fi}')

print(f'Round trip: {jnp.array_equal(pixels, hp.xyf2pix(nside, x, y, face))}')

In [ ]:
nside_faces = 16
_, _, faces = hp.pix2xyf(nside_faces, jnp.arange(hp.nside2npix(nside_faces)))

mollview(faces, title='Base face of each pixel', cmap='tab20c', cbar=False)
# Label each face at the center of its base pixel (nside = 1, NESTED index = face)
face_theta, face_phi = hp.pix2ang(1, jnp.arange(12), nest=True)
for t, p, face in zip(face_theta, face_phi, range(12)):
    projtext(t, p, str(face), ha='center', va='center', fontsize=12, fontweight='bold')
plt.show()

## Astronomical coordinates

The angles of HEALPix are the colatitude $\theta$, in $[0, \pi]$ from the North pole, and the
longitude $\phi$, in radians. With `lonlat=True`, the functions take and return instead the
longitude and the latitude in degrees, such as the right ascension $\alpha$ and the
declination $\delta$ of equatorial coordinates: $\theta = 90° - \delta$ and $\phi = \alpha$.

In [ ]:
messier_coords = {
    'M1 (Crab Nebula)': (83.6331, 22.0145),
    'M31 (Andromeda)': (10.6847, 41.2689),
    'M42 (Orion Nebula)': (83.8221, -5.3911),
}

nside = 128
for name, (ra, dec) in messier_coords.items():
    pixel = hp.ang2pix(nside, ra, dec, lonlat=True)
    print(f'{name}: RA = {ra:.2f}°, Dec = {dec:.2f}° → pixel {pixel}')

In [ ]:
mollview(
    jnp.zeros(hp.nside2npix(1)),
    title='Messier objects (equatorial coordinates)',
    cmap='Greys',
    min=0,
    max=1,
    cbar=False,
)
graticule(dpar=30, dmer=60, alpha=0.4)
for name, (ra, dec) in messier_coords.items():
    projscatter(ra, dec, lonlat=True, s=40, color='#2a78d6')
    projtext(ra, dec, '  ' + name.split()[0], lonlat=True, fontsize=10)
plt.show()

## Interpolation

A map has a value per pixel. `get_interp_weights` returns, for any direction, the 4 nearest
pixels and their weights, and `get_interp_val` interpolates the map bilinearly between them.
The interpolation points are marked with crosses:

In [ ]:
nside = 64
theta, phi = hp.pix2ang(nside, jnp.arange(hp.nside2npix(nside)))
test_map = jnp.sin(theta) * jnp.cos(phi)

theta_interp = jnp.array([0.5, 1.0, 1.5])
phi_interp = jnp.array([0.0, 1.0, 2.0])
neighbours, weights = hp.get_interp_weights(nside, theta_interp, phi_interp)
interpolated_values = hp.get_interp_val(test_map, theta_interp, phi_interp)

mollview(test_map, title='sin(θ) cos(φ)', cmap='RdBu_r', min=-1, max=1)
projscatter(theta_interp, phi_interp, marker='x', s=80, color='black')
plt.show()

print(f'Interpolated values: {interpolated_values}')
print(f'Exact values:        {jnp.sin(theta_interp) * jnp.cos(phi_interp)}')

## Choosing the resolution

`nside` sets the number of pixels, $12\,n_\mathrm{side}^2$, the order
$\log_2 n_\mathrm{side}$ and the size of the pixels:

In [ ]:
for nside in [64, 256, 1024, 2048]:
    print(
        f'nside = {nside:4d}: order {hp.nside2order(nside):2d}, {hp.nside2npix(nside):8d} pixels of '
        f"{hp.nside2resol(nside, arcmin=True):5.2f}', {hp.nside2pixarea(nside, degrees=True) * 3600:6.1f} arcmin²"
    )

The resolution is usually set by the beam of the instrument: a common rule of thumb is to
sample its full width at half maximum (FWHM) with at least 2 pixels: with `nside` a power of 2,
there are then 2 to 4 pixels per FWHM. For example, Planck maps its 5' beam with
`nside = 2048`. The band limit of the spherical harmonic transforms then follows from `nside`:
by default, `lmax = 3 * nside - 1`.

In [ ]:
def nside_for_beam(fwhm_arcmin, pixels_per_fwhm=2):
    """Smallest nside whose pixels sample the beam FWHM with at least the given number of pixels."""
    nside = 1
    while hp.nside2resol(nside, arcmin=True) > fwhm_arcmin / pixels_per_fwhm:
        nside *= 2
    return nside


for fwhm in [60, 30, 10, 5, 1]:  # arcminutes
    nside = nside_for_beam(fwhm)
    resolution = hp.nside2resol(nside, arcmin=True)
    print(f"FWHM {fwhm:2d}': nside = {nside:4d}, pixel size {resolution:5.2f}', lmax = {3 * nside - 1}")

As in healpy, `isnsideok` accepts any positive integer for the RING order. The pixel functions
of jax-healpy require a power of 2, as the NESTED order does: check it with `nest=True`.

In [ ]:
for nside in [1, 3, 64, 127, 128]:
    print(f'nside = {nside:3d}: RING {hp.isnsideok(nside)}, power of 2 {hp.isnsideok(nside, nest=True)}')